# 01 — Setup, protocol and ledger / 交易設定、評估協定與研究帳本

**EN** The hypothesis, trading setup v1 and evaluation protocol, all committed before any backtest; the research ledger that feeds the Deflated Sharpe Ratio; the single holdout touch.

**中文** 假說、交易設定 v1 與評估協定（皆在任何回測前 commit）；計算 Deflated Sharpe 所用的研究帳本；holdout 唯一一次的讀取紀錄。

In [1]:
# Locate the repository root and load the library (works from the root or notebooks/)
# 找到 repo 根目錄並載入函式庫（從根目錄或 notebooks/ 執行皆可）
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "setup.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200, "display.max_columns", 30, "display.precision", 4)


def show(fig, width=820):
    """Show a committed figure / 顯示已產出的圖"""
    display(Image(filename=str(ROOT / "reports" / "figures" / fig), width=width))


def sh(cmd):
    """Run a repository command from the root and print its output / 在根目錄執行指令並印出結果"""
    r = subprocess.run(cmd, shell=True, cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
    print(r.stdout[-4000:], r.stderr[-2000:])
    return r.returncode

## Hypothesis
**EN** Crowded leveraged longs pay funding and later unwind, so funding and positioning data should rank next-week returns across the 100 most liquid Binance USDT-M perpetuals. Edge source: risk compensation and funding constraints (Table 6.1).

## 假說
**中文** 擁擠的槓桿多頭持續支付資金費、之後被迫平倉，因此資金費與部位資料應能排序 Binance 前 100 大 USDT 永續合約的下週報酬。優勢來源：風險補償與資金限制（Table 6.1）。

In [2]:
# setup.yaml is the frozen v1 contract / setup.yaml 是凍結的 v1 合約
setup = yaml.safe_load(open("setup.yaml", encoding="utf-8"))
print("version / 版本:", setup["setup_version"])
for k in ("universe", "decision", "mapping", "costs"):
    print(f"\n[{k}]")
    print(yaml.safe_dump(setup.get(k), allow_unicode=True, sort_keys=False)[:900])

version / 版本: v1

[universe]
rule: top_n_by_trailing_dollar_volume
top_n: 100
rank_window_days: 30
min_median_quote_volume_usd: 5000000
min_listing_age_days: 60
min_names_to_trade: 30
exclude_symbols:
- USDCUSDT
- PAXGUSDT
- XAUTUSDT
- DEFIUSDT
- BTCDOMUSDT
leavers: included
identifier_policy: 'A ticker that was delisted and relisted is a new contract from
  its first price bar; funding settlements before that bar belong to the old contract
  and are dropped. Renames (MATIC->POL, FTM->S, RNDR->RENDER) are separate contracts,
  not spliced.

  '


[decision]
cadence: daily
snapshot: 00:00 UTC of decision date d
admissible_information: records with available_at <= d 00:00 UTC
execution: open of daily bar d (00:00 UTC), same instant as the snapshot
execution_delay_robustness: 1 day (fill at the close of bar d)


[mapping]
position_state_space: long_short
selection: top 20% scores long, bottom 20% short
weighting: equal_weight
rebalance: 7 staggered tranches; tranche (d mod 7) rebalances o

## Evaluation protocol
**EN** Walk-forward folds with a 7-day purge; the holdout dates; the registered confirmatory test.

## 評估協定
**中文** 含 7 天 purge 的 walk-forward 切分、holdout 日期、事先登記的確認性檢定。

In [3]:
# Fold dates come from the library, not typed by hand / 切分日期由程式產生，不是手打
from crowding.splits import fold_table
folds = fold_table()
print(len(folds), "folds / 個 fold")
display(folds.head(3)); display(folds.tail(2))
ev = setup["evaluation"]
print("holdout / 封存期:", ev["holdout"]["start"], "→", ev["holdout"]["end"],
      "| shadow start / 影子期起點:", ev["forward_shadow"]["start"])
print("confirmatory test / 確認性檢定:", yaml.safe_load(open("config/holdout.yaml", encoding="utf-8")).get("confirmatory"))

32 folds / 個 fold


,fold,train_start,val_start,val_end,inner_val_start
0,0,2022-03-01,2023-03-01,2023-03-31,2022-12-01
1,1,2022-03-01,2023-04-01,2023-04-30,2023-01-01
2,2,2022-03-01,2023-05-01,2023-05-31,2023-02-01


,fold,train_start,val_start,val_end,inner_val_start
30,30,2022-03-01,2025-09-01,2025-09-30,2025-06-01
31,31,2022-03-01,2025-10-01,2025-10-24,2025-07-01


holdout / 封存期: 2025-10-25 → 2026-10-24 | shadow start / 影子期起點: 2026-10-25
confirmatory test / 確認性檢定: {'model': 'ridge', 'test': 'holdout mean daily Spearman IC (ridge vs y_total_7d), Newey-West 95% CI with lags >= 7', 'pass_if': 'CI lower bound > 0', 'secondary': 'ridge, equal weight, setup-v1 mapping and costs: holdout net Sharpe with 95% CI and Deflated Sharpe; reported, not a pass/fail gate'}


## Research ledger
**EN** Every trial family and its budget. Families registered after the holdout (setup v2) do not count toward the holdout's deflation.

## 研究帳本
**中文** 每個試驗家族與其預算。holdout 之後才登記的家族（setup v2）不計入 holdout 的 Sharpe 平減。

In [4]:
# Trial counts used by the Deflated Sharpe Ratio / Deflated Sharpe 使用的試驗次數
from crowding.ledger import trials_run
led = pd.read_csv("ledger/research_ledger.csv")
display(led[["family", "trial", "n_trials", "status", "after_holdout"]])
print("trials before the holdout / holdout 前試驗數:", trials_run(before_holdout=True))
print("all trials / 全部試驗數:", trials_run())

,family,trial,n_trials,status,after_holdout
0,F0_prior,xs_explore_v1,72,closed,False
1,F0_prior,xs_explore_v2,63,closed,False
2,F0_prior,xs_cap20000,96,closed,False
3,F0_prior,smallcap_system,12,closed,False
4,F0_prior,run_momentum,16,closed,False
5,F0_prior,run_carry,9,closed,False
6,F0_prior,analyze_alpha,36,closed,False
7,F0_prior,lp_pools,27,closed,False
8,F1_rule,rule_carry,1,registered,False
9,F2_features,triage,24,registered,False


trials before the holdout / holdout 前試驗數: 454
all trials / 全部試驗數: 456


In [5]:
# The holdout was read exactly once; shadow reads are logged too / holdout 只讀過一次，影子期讀取也有紀錄
display(pd.read_csv("ledger/holdout_touches.csv"))
log = pd.read_csv("run_log.csv")
print(len(log), "runs in run_log.csv / 筆執行紀錄"); display(log[["run_id", "family", "trial", "decision"]].tail(8))

,utc,period,git_commit,reason
0,2026-10-27T04:50:09+00:00,holdout,0ba5579,"W6 single holdout run, config/holdout.yaml fro..."
1,2026-10-27T04:50:17+00:00,shadow,0ba5579,frozen ridge shadow predictions; no labels exist
2,2026-10-27T08:02:05+00:00,shadow,fa5b602,feature coverage for the committed shadow pred...
3,2026-10-27T08:12:00+00:00,shadow,eaa5df0,registered shadow test of setup v2


42 runs in run_log.csv / 筆執行紀錄


,run_id,family,trial,decision
34,R0035,F9_setup_v2,rule_carry v2a,exploratory (not confirmatory)
35,R0036,F9_setup_v2,shadow_test,pending
36,R0037,F9_setup_v2,ridge v1,exploratory (not confirmatory)
37,R0038,F9_setup_v2,ridge v2a,exploratory (not confirmatory)
38,R0039,F9_setup_v2,ridge v2b,exploratory (not confirmatory)
39,R0040,F9_setup_v2,rule_carry v1,exploratory (not confirmatory)
40,R0041,F9_setup_v2,rule_carry v2a,exploratory (not confirmatory)
41,R0042,F9_setup_v2,mc_robustness,descriptive (not a trial)
